In [ ]:
# ================================
# WEEK 4 - FINAL MODEL TRAINING
# ================================

# 1. Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import joblib

# ================================
# 2. Load Dataset
# ================================

# 👉 CHANGE THIS PATH TO YOUR FILE
data = pd.read_csv("earthquake_data.csv")

print("Dataset Loaded Successfully!")
print(data.head())

# ================================
# 3. Data Preprocessing
# ================================

# Drop unnecessary columns if present
drop_cols = ['time', 'place', 'id']
for col in drop_cols:
    if col in data.columns:
        data = data.drop(columns=[col])

# Handle missing values
data = data.dropna()

print("After Cleaning:", data.shape)

# ================================
# 4. Feature Selection
# ================================

# Selecting important features
features = ['latitude', 'longitude', 'depth_km', 'sig']

# Target column (modify if needed)
target = 'mag'

# Convert magnitude into categories
def categorize_magnitude(mag):
    if mag < 4:
        return "Low"
    elif mag < 6:
        return "Medium"
    else:
        return "High"

data[target] = data[target].apply(categorize_magnitude)

# Encode labels
le = LabelEncoder()
data[target] = le.fit_transform(data[target])

# ================================
# 5. Split Data
# ================================

X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", X_train.shape)

# ================================
# 6. Train Model (Gradient Boosting)
# ================================

model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3
)

model.fit(X_train, y_train)

print("Model Training Completed!")

# ================================
# 7. Evaluation
# ================================

y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ================================
# 8. Save Model
# ================================

joblib.dump(model, "earthquake_model.pkl")
joblib.dump(le, "label_encoder.pkl")

print("Model Saved Successfully!")

# ================================
# 9. Sample Prediction
# ================================

# Example input
sample = pd.DataFrame(
    [[7.02, 10.0, 9.0, 8.0]],
    columns=['latitude','longitude','depth_km','sig']
)

prediction = model.predict(sample)
predicted_label = le.inverse_transform(prediction)

print("\nPredicted Magnitude Category:", predicted_label[0])